In [1]:
# ============================================================
# YOLOv26-large (yolo26l)  |  TRAIN + DANH GIA TEST trong 1 CELL
# Truoc khi chay:
#   (1) Add Input: dataset chua YOLO_Final_Dataset  ->  sua KAGGLE_DATA_PATH ben duoi
#   (2) Settings: Accelerator = GPU T4 x2   |   Internet = On
# ============================================================
!pip install -q ultralytics

from ultralytics import YOLO
import glob, os

# -------- CAU HINH --------
KAGGLE_DATA_PATH = '/kaggle/input/datasets/tungtungtungtung/final-dataset-yolo/YOLO_Final_Dataset'  # <-- SUA cho khop
WEIGHTS  = 'yolo26l.pt'
RUN_NAME = 'yolo26l_telecom'
BATCH    = 8

# tao dataset.yaml
yaml_content = f"""
path: {KAGGLE_DATA_PATH}
train: train/images
val: val/images
test: test/images

nc: 6
names: ['anten-4G', 'anten-5G', 'none', 'rrh', 'rru', 'viba']
"""
YAML_PATH = '/kaggle/working/dataset.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content.strip())

# -------- TRAIN --------
model = YOLO(WEIGHTS)
model.train(
    data=YAML_PATH, imgsz=640, device=[0, 1], batch=BATCH, workers=8, amp=True,
    epochs=150, patience=25, pretrained=True, optimizer='auto', lr0=0.01,
    cos_lr=True, close_mosaic=10, seed=42, deterministic=True,
    project='telecom_vision_project', name=RUN_NAME, save_period=10,
)
print('🎉 Huan luyen hoan tat! Model luu trong runs/detect/telecom_vision_project/' + RUN_NAME)

# -------- DANH GIA TEST (boc try/except -> KHONG bao gio lam hong run khi save) --------
try:
    cands = glob.glob(f'/kaggle/working/**/{RUN_NAME}*/weights/best.pt', recursive=True)
    best_path = max(cands, key=os.path.getmtime)
    print('Best weights:', best_path)
    mt = YOLO(best_path).val(data=YAML_PATH, split='test', imgsz=640)
    print(f'>>> Test mAP50-95: {mt.box.map:.4f} | mAP50: {mt.box.map50:.4f} | mAP75: {mt.box.map75:.4f}')
except Exception as e:
    print('Bo qua danh gia test (model da train van duoc luu an toan):', repr(e))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 83.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23